# 11 - Random-intercept models

The Results chapter is descriptive: every claim it makes compares point
estimates - a heat-map cell against another cell, a bar against the bar beside
it - with no standard error attached. This notebook adds the inferential layer,
using a random intercept for the model, and checks that the instrument is
appropriate to this data before trusting what it says.

Sections 1-3 fix what is being modelled, build the frame, and establish that the
random intercept earns its place. Sections 4-7 take the four research questions
one at a time, each stating the question, the answer the chapter currently
gives, the model output, and what that output does to the answer. Section 8
collects the verdict.

---

## 1. The random intercept, in this experiment's variables

The experiment varies five things and holds the corpus fixed: **regime**
(document or sentence), **phase** (prompt selection or test), **model**,
**language**, and **prompt variant** - with **temperature** swept inside the
test phase. Section 2 puts one row per combination: 207 runs.

Those runs are not independent, and that is what the random intercept is for.
Each model contributes a whole block of them, and the model's general capability
is a component of every score in its block: `gemini` scores above the field
under every prompt and in both languages, `amalia` below it under all of them.
That between-model capability is a nuisance - RQ1 asks about the prompt, not
about which model is strongest - and it is *larger* than the effects being
tested, so leaving it in the residual buries them.

A random intercept gives each model $m$ its own offset $u_m$ and estimates the
experimental effects net of it:

$$
y_{im} = \mathbf{x}_{im}^{\top}\boldsymbol{\beta} + u_m + \varepsilon_{im},
\qquad u_m \sim \mathcal{N}(0, \sigma^2_{\text{model}}).
$$

| Experimental variable | Role in the model |
| --- | --- |
| Prompt variant, decomposed into `has_def`, `has_exp`, `task` | Fixed effects - RQ1 |
| Language (PT / EN) | Fixed effect - RQ2, RQ4 |
| Temperature (test phase) | Fixed effect - RQ1 |
| Regime (document / sentence) | Fixed effect - RQ1, RQ4 |
| Phase | Restriction: the two phases are fitted separately |
| **Model** | **Grouping factor - the random intercept** |
| `f1_ext`, `f1_cls`, `accuracy` of one run | Response |

This is what lets a prompt effect be read as *the effect on a typical model*
rather than an average that whichever model scores highest happens to dominate.
It also matches the chapter's own reading of Fig. `prompt-sensitivity`:
sensitivity is a property of the model, so the model is what the design should
be pooled over.

One caveat, stated once. With nine or ten models the variance component rests on
few groups, and the responses are proportions rather than unbounded Gaussians,
so the intervals below are approximations. They are used to separate effects
that are clearly real from effects that are indistinguishable from noise, not to
support fine numerical claims about their size.

---

## 2. Consolidated results frame

One row per experimental run: a (regime, phase, language, model, prompt variant,
temperature) combination, with the metrics the Results chapter reports.

Counts come from the `tp`/`fp`/`fn` bookings of `detailed_results.csv`, so all
three result trees are summarised by the same rule; `f1_r` comes from
`results.csv`. Location subtypes are collapsed into `Loc`, which is the view the
chapter reports.

In [1]:
from pathlib import Path
import sys
import warnings

import numpy as np
import pandas as pd

sys.path.insert(0, str(Path.cwd()))   # run from notebooks/results/
import _common as common

import statsmodels.api as sm
import statsmodels.formula.api as smf
from scipy import stats

warnings.filterwarnings("ignore")
pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 40)

In [2]:
# (regime, phase, results/ subtree). The sentence-level runs live under `cls/`,
# which carries the assigned types; the parallel `ext/` tree is the same runs
# scored without them.
SOURCES = [
    ("document", "prompt_selection", "prompt_selection"),
    ("document", "test", "test"),
    ("sentence", "test", "sentence_level/cls"),
]


def summarise_runs(regime: str, phase: str, stage: str) -> pd.DataFrame:
    """One row per (language, model, template) run of one results subtree."""
    detailed = pd.concat(
        [common.load_detailed(stage, language).assign(language=language) for language in common.LANGUAGES],
        ignore_index=True,
    )
    summary = common.compute_extraction_classification_metrics(
        common.coarsen_types(detailed), group_cols=["language", "model", "template"]
    )

    relaxed = pd.concat([common.load_results(stage, language) for language in common.LANGUAGES],
                        ignore_index=True)
    summary = summary.merge(
        relaxed[["language", "model", "template", "f1_r_macro"]],
        on=["language", "model", "template"], how="left",
    )

    summary.insert(0, "regime", regime)
    summary.insert(1, "phase", phase)
    return summary


def build_experiment_frame() -> pd.DataFrame:
    """The consolidated frame: every run of every experiment, one row each.

    `prompt` is the prompt variant with the decoding configuration stripped off,
    renamed to the chapter's `sent_*` labels in the sentence-level regime.
    `has_def` / `has_exp` / `task` decompose it into the levers under study;
    `task` is left missing for the sentence-level runs, whose three-stage
    pipeline always classifies and so does not sit on the ext/cls contrast.
    """
    frame = pd.concat([summarise_runs(*source) for source in SOURCES], ignore_index=True)
    frame = common.add_run_config_columns(frame)

    is_sentence = frame["regime"] == "sentence"
    frame["prompt"] = np.where(
        is_sentence,
        "sent_" + frame["base_template"].str.split("_", n=1).str[1],
        frame["base_template"],
    )
    frame["has_def"] = frame["base_template"].str.contains("_def").astype(int)
    frame["has_exp"] = frame["base_template"].str.contains("_exp").astype(int)
    task = np.where(frame["base_template"].str.startswith("cls"), "cls", "ext")
    frame["task"] = pd.Series(task, index=frame.index).where(~is_sentence)
    frame["lang"] = frame["language"].map({"portuguese": "PT", "english": "EN"})

    frame["n_annotated"] = frame["tp_ext"] + frame["fn_ext"]
    frame["extracted"] = frame["tp_ext"]
    frame["typed"] = frame["tp_cls"]
    frame["accuracy"] = frame["typed"] / frame["extracted"].replace(0, np.nan)

    columns = [
        "regime", "phase", "language", "lang", "model", "prompt", "template",
        "temperature", "has_def", "has_exp", "task",
        "n_annotated", "extracted", "typed", "accuracy",
        "precision_ext", "recall_ext", "f1_ext", "f1_cls", "f1_r",
    ]
    frame = frame.rename(columns={"f1_r_macro": "f1_r"})[columns]
    return frame.sort_values(
        ["regime", "phase", "language", "model", "prompt", "temperature"]
    ).reset_index(drop=True)


experiments = build_experiment_frame()
experiments.to_csv("all_experiments.csv", index=False)
print(f"{len(experiments)} runs x {experiments.shape[1]} columns")
print(experiments.groupby(["regime", "phase", "language"]).size().rename("runs").to_string())

207 runs x 20 columns
regime    phase             language  
document  prompt_selection  english       80
                            portuguese    72
          test              english       25
                            portuguese    26
sentence  test              english        2
                            portuguese     2


The frame is only useful if it reproduces what the chapter already reports, so
published cells from three of its tables are asserted against it rather than
eyeballed.

In [3]:
def run(**keys) -> pd.Series:
    """The single row matching `keys`."""
    mask = np.ones(len(experiments), dtype=bool)
    for column, value in keys.items():
        mask &= experiments[column] == value
    matched = experiments[mask]
    assert len(matched) == 1, f"{keys} matched {len(matched)} rows"
    return matched.iloc[0]


checks = [
    # tab:participant-funnel -- gemini, Portuguese, cls_def_exp
    (run(regime="document", phase="prompt_selection", language="portuguese",
         model="gemini", prompt="cls_def_exp"),
     dict(f1_ext=0.564, f1_cls=0.497, extracted=293, typed=258, accuracy=0.881, n_annotated=523)),
    # tab:test-participant-funnel -- gemini, Portuguese, T=1
    (run(regime="document", phase="test", language="portuguese", model="gemini",
         prompt="cls_def_exp", temperature=1.0),
     dict(f1_ext=0.553, f1_cls=0.493, extracted=1434, typed=1279, accuracy=0.892, n_annotated=2580)),
    # tab:sentence-cascade -- qwen3_14b
    (run(regime="sentence", language="portuguese", model="qwen3_14b", prompt="sent_exp"),
     dict(f1_ext=0.451, extracted=1031, typed=927, accuracy=0.899)),
    (run(regime="sentence", language="english", model="qwen3_14b", prompt="sent_def_exp"),
     dict(f1_ext=0.394, extracted=867, typed=747, accuracy=0.862)),
]

for row, expected in checks:
    for column, value in expected.items():
        assert round(float(row[column]), 3) == value, (
            f"{row['regime']}/{row['model']}/{row['prompt']}: "
            f"{column} is {row[column]!r}, chapter reports {value!r}"
        )
print(f"all {sum(len(e) for _, e in checks)} published figures reproduced")

all 20 published figures reproduced


`tab:test-participant-funnel` reports `euro_llm_9b` in English on `cls_exp` at
$T=0.6$ (0.218 / 0.154, 526 located, 372 typed). No such run exists in
`results/test/english/` - the only English `euro_llm_9b` runs on disk are
`cls_def_exp` at the three temperatures. The results were re-run after that
table was written, so **the table is stale for that row**.

In [4]:
print(experiments[(experiments["regime"] == "document")
                  & (experiments["phase"] == "test")
                  & (experiments["language"] == "english")
                  & (experiments["model"] == "euro_llm_9b")]
      [["prompt", "temperature", "f1_ext", "f1_cls", "extracted", "typed", "accuracy"]]
      .round(3).to_string(index=False))

     prompt  temperature  f1_ext  f1_cls  extracted  typed  accuracy
cls_def_exp          0.3   0.229   0.172        518  388.0     0.749
cls_def_exp          0.6   0.242   0.169        591  412.0     0.697
cls_def_exp          1.0   0.160   0.105        377  247.0     0.655


---

## 3. Does the random intercept earn its place?

Fitting a mixed model is only justified if the grouping factor absorbs something
real. Three checks, on the prompt-selection panel: how much of the variance sits
between models, whether the random intercept beats the pooled model on
likelihood, and what it does to the standard errors.

In [5]:
def fit_ri(formula: str, data: pd.DataFrame, groups: str = "model", reml: bool = True):
    """Random-intercept fit, trying optimisers in turn until one converges.

    The panels are small and several specifications sit near the boundary of the
    parameter space, where lbfgs occasionally fails outright; the alternatives
    recover those cases without changing the fitted values where lbfgs succeeds.
    """
    model = smf.mixedlm(formula, data=data, groups=data[groups])
    error = None
    for method in ("lbfgs", "bfgs", "powell"):
        try:
            return model.fit(reml=reml, method=method)
        except Exception as exc:      # noqa: BLE001 - retried with the next optimiser
            error = exc
    raise error


def coefficients(result, label: str = "") -> pd.DataFrame:
    """Tidy coefficient table: estimate, standard error, 95% CI and p-value.

    The `Group Var` row statsmodels appends is dropped - it is reported as a
    variance component by `icc` below, and its Wald interval is not meaningful
    for a parameter bounded at zero.
    """
    interval = result.conf_int()
    table = pd.DataFrame({
        "coef": result.params, "se": result.bse,
        "ci_lo": interval[0], "ci_hi": interval[1], "p": result.pvalues,
    }).drop(index="Group Var", errors="ignore")
    if label:
        print(f"--- {label}  (n={int(result.nobs)}, models={result.model.n_groups})")
    return table.round(4)


def icc(result) -> float:
    """Share of the residual-plus-group variance sitting between models.

    `cov_re` holds the group variance on the response scale; the `Group Var`
    entry of `params` is that variance divided by the residual, so it must not
    be used here.
    """
    between = float(result.cov_re.iloc[0, 0])
    return between / (between + float(result.scale))


SELECTION = experiments[(experiments["regime"] == "document")
                        & (experiments["phase"] == "prompt_selection")].copy()
TEST = experiments[(experiments["regime"] == "document")
                   & (experiments["phase"] == "test")].copy()

PROMPT_FORMULA = ("f1_ext ~ has_exp + has_def + C(task, Treatment('ext'))"
                  " + C(lang, Treatment('PT'))")

In [6]:
warrant = fit_ri(PROMPT_FORMULA, SELECTION)
pooled = smf.ols(PROMPT_FORMULA, data=SELECTION).fit()

print(f"variance between models : {float(warrant.cov_re.iloc[0, 0]):.5f}")
print(f"residual variance       : {float(warrant.scale):.5f}")
print(f"ICC                     : {icc(warrant):.3f}")

# Likelihood-ratio test against the pooled model. Both are fitted by ML, since
# REML likelihoods are not comparable with the OLS one. The null puts the
# variance on the boundary of the parameter space, so the p-value takes the
# standard 50:50 mixture correction.
warrant_ml = fit_ri(PROMPT_FORMULA, SELECTION, reml=False)
statistic = 2 * (warrant_ml.llf - pooled.llf)
print(f"\nLRT vs pooled OLS       : chi2={statistic:.1f}, p={0.5 * stats.chi2.sf(statistic, 1):.2e}")

comparison = pd.DataFrame({"ols_se": pooled.bse, "ri_se": warrant.bse}).dropna()
comparison["ratio"] = comparison["ri_se"] / comparison["ols_se"]
print("\nstandard errors, random intercept against pooled OLS:")
print(comparison.round(4).to_string())

print(f"\nresiduals, Shapiro-Wilk : W={stats.shapiro(warrant.resid)[0]:.3f}, "
      f"p={stats.shapiro(warrant.resid)[1]:.4f}")

variance between models : 0.00607
residual variance       : 0.00341
ICC                     : 0.640

LRT vs pooled OLS       : chi2=118.8, p=5.76e-28

standard errors, random intercept against pooled OLS:
                                  ols_se   ri_se   ratio
C(lang, Treatment('PT'))[T.EN]    0.0156  0.0097  0.6214
C(task, Treatment('ext'))[T.cls]  0.0156  0.0095  0.6066
Intercept                         0.0177  0.0270  1.5262
has_def                           0.0156  0.0095  0.6066
has_exp                           0.0156  0.0095  0.6066

residuals, Shapiro-Wilk : W=0.978, p=0.0144


**The random intercept is warranted.** 64% of the variance in span-extraction F1
sits *between* models rather than within them, and the likelihood-ratio test
against the pooled model is decisive ($\chi^2 = 118.8$, $p \approx 6\times
10^{-28}$).

The standard errors show what pooling was costing. For every within-model
effect - the prompt components, the language - the random-intercept standard
error is about **0.61x** the OLS one: pooling left the large between-model
capability differences in the residual, which inflated the noise estimate and
made real prompt effects look less certain than they are. The intercept moves
the other way (1.53x), which is correct: the average score across models is
estimated from ten models, not from 152 independent runs.

Residuals are mildly non-normal ($W = 0.978$, $p = 0.014$), as expected for a bounded
proportion. This is why the intervals below are read as separating real effects
from noise rather than as exact numerical claims.

---

## 4. RQ1 - prompt sensitivity across languages, and span against attribute

> **RQ1.** How does prompt sensitivity in LLM-based participant extraction vary
> between Portuguese and English, and is the stability of the method impacted
> when moving from span detection to attribute (entity types) classification?

**The chapter's descriptive answer.** Sensitivity is a property of the *model*,
not the language: the two bars of a given model stand at similar heights while
models differ markedly. The prompt nevertheless governs span extraction
strongly, and one component does the work - in both languages every variant
carrying the one-shot example scores above every variant that does not, `gemini`
moving from 0.106 under `ext` to 0.564 under `cls_def_exp`. The definition is a
weak lever, and task is "a wash". At the attribute level the method is
distinctly more stable: the four `cls_*` variants give accuracies at much the
same height, above 0.70 throughout.

In [7]:
# Span extraction: do the prompt components act differently in the two languages?
rq1_extraction = fit_ri(
    "f1_ext ~ (has_exp + has_def) * C(lang, Treatment('PT')) + C(task, Treatment('ext'))",
    SELECTION,
)
print(coefficients(rq1_extraction, "RQ1a  span extraction, prompt x language").to_string())
print(f"ICC = {icc(rq1_extraction):.3f}")

--- RQ1a  span extraction, prompt x language  (n=152, models=10)
                                          coef      se   ci_lo   ci_hi       p
Intercept                               0.0767  0.0279  0.0220  0.1313  0.0060
C(lang, Treatment('PT'))[T.EN]          0.0117  0.0166 -0.0209  0.0443  0.4816
C(task, Treatment('ext'))[T.cls]        0.0467  0.0095  0.0281  0.0653  0.0000
has_exp                                 0.1333  0.0138  0.1062  0.1604  0.0000
has_exp:C(lang, Treatment('PT'))[T.EN]  0.0206  0.0190 -0.0167  0.0579  0.2795
has_def                                 0.0077  0.0138 -0.0194  0.0347  0.5782
has_def:C(lang, Treatment('PT'))[T.EN]  0.0083  0.0190 -0.0290  0.0456  0.6637
ICC = 0.639


In [8]:
# Task task: the chapter calls it a wash, but its own prose says the two
# tasks separate among the *bare* variants and converge once the example is
# present. That is an interaction, so it is fitted as one.
rq1_task = fit_ri(
    "f1_ext ~ has_exp * C(task, Treatment('ext')) + has_def + C(lang, Treatment('PT'))",
    SELECTION,
)
print(coefficients(rq1_task, "RQ1b  task x example").to_string())

print("\nsimple effect of the cls task, either side of the example:")
task_simple = {}
for label, subset in (("without the example", SELECTION[SELECTION["has_exp"] == 0]),
                      ("with the example", SELECTION[SELECTION["has_exp"] == 1])):
    simple = fit_ri("f1_ext ~ C(task, Treatment('ext')) + has_def + C(lang, Treatment('PT'))",
                    subset)
    task_simple[label] = simple
    row = coefficients(simple).loc["C(task, Treatment('ext'))[T.cls]"]
    print(f"  {label:<20} {row['coef']:+.4f}  [{row['ci_lo']:+.4f}, {row['ci_hi']:+.4f}]  p={row['p']:.4f}")

--- RQ1b  task x example  (n=152, models=10)


                                            coef      se   ci_lo   ci_hi       p
Intercept                                 0.0558  0.0273  0.0024  0.1093  0.0405
C(task, Treatment('ext'))[T.cls]          0.0731  0.0131  0.0475  0.0987  0.0000
C(lang, Treatment('PT'))[T.EN]            0.0261  0.0095  0.0075  0.0447  0.0059
has_exp                                   0.1706  0.0131  0.1449  0.1962  0.0000
has_exp:C(task, Treatment('ext'))[T.cls] -0.0529  0.0185 -0.0891 -0.0166  0.0042
has_def                                   0.0120  0.0092 -0.0061  0.0301  0.1930

simple effect of the cls task, either side of the example:
  without the example  +0.0731  [+0.0554, +0.0908]  p=0.0000
  with the example     +0.0203  [+0.0026, +0.0380]  p=0.0248


In [9]:
# Attribute level. `accuracy` is conditional on a run's own extraction count, so
# runs resting on a handful of located participants carry no information about
# typing -- the chapter makes the same point about its 2-of-4 cells. Runs with
# fewer than 20 located participants are therefore excluded.
MIN_EXTRACTED = 20
typed_runs = SELECTION[SELECTION["accuracy"].notna()
                       & (SELECTION["extracted"] >= MIN_EXTRACTED)]

rq1_typing = fit_ri("accuracy ~ has_exp + has_def + C(lang, Treatment('PT'))", typed_runs)
print(coefficients(rq1_typing, f"RQ1c  typing accuracy (extracted >= {MIN_EXTRACTED})").to_string())

print(f"\nresponse levels: mean f1_ext = {SELECTION['f1_ext'].mean():.3f}, "
      f"mean typing accuracy = {typed_runs['accuracy'].mean():.3f}")

--- RQ1c  typing accuracy (extracted >= 20)  (n=70, models=10)
                                  coef      se   ci_lo   ci_hi       p
Intercept                       0.9186  0.0162  0.8868  0.9503  0.0000
C(lang, Treatment('PT'))[T.EN] -0.0385  0.0102 -0.0585 -0.0185  0.0002
has_exp                        -0.0477  0.0102 -0.0676 -0.0278  0.0000
has_def                         0.0104  0.0100 -0.0091  0.0299  0.2950

response levels: mean f1_ext = 0.186, mean typing accuracy = 0.879


**Supported, with one refinement.**

*The example dominates, and identically in both languages.* Carrying the
one-shot example is worth **+0.133 F1** [0.106, 0.160], the largest effect
anywhere in this notebook. Its interaction with language is +0.021
[-0.017, 0.058], $p = 0.28$: the example is worth the same in Portuguese and in
English. The definition is +0.008 [-0.019, 0.035], $p = 0.58$ - indistinguishable
from zero, which is a stronger statement than the chapter's "weak lever", and
its language interaction is null too. So prompt sensitivity does not differ
between the languages, exactly as claimed.

*Task is not a wash - it is conditional.* The `cls` task beats `ext`
by **+0.073** [0.055, 0.091] among the bare variants, and by only **+0.020**
[0.003, 0.038] once the example is present; the interaction is significant
($-0.053$, $p = 0.004$). The chapter's own prose says this, but the summary claim
that task does not matter is too strong: asking for the type alongside the
span helps substantially when the prompt has nothing else to go on, and helps
slightly even when it does.

*The attribute level is the more stable, though not perfectly flat.* Typing
accuracy sits at 0.919 and the example moves it by **-0.048** [-0.068, -0.028] -
a real decline, but against a response that sits far higher than F1-ext (mean
0.88 against 0.19) and in the opposite direction to its effect on extraction. The
prompt does not buy typing accuracy the way it buys extraction; the small
decline is consistent with the chapter's own reading, that example-bearing runs
locate many more participants and so reach further into the harder part of the
corpus. RQ1's second clause stands.

---

## 5. RQ2 - are the errors consistent across the two languages?

> **RQ2.** When an LLM extracts participants from parallel Portuguese and
> English versions of the same news article, are extraction errors consistent
> across languages (suggesting conceptual limitations) or language-specific
> (suggesting surface-level processing difficulties)?

**The chapter's descriptive answer.** Consistent, at every granularity examined.
The outcome mixes differ far more between models than between the two languages
of one model; the word-level evaluation puts the failure at the same place in
both - the opening slot of the participant, where the error is almost always a
determiner. The exceptions "attach to particular models rather than to a
language". The errors belong to the method.

In [10]:
# If the limitation is conceptual, a configuration's Portuguese score should
# predict its English score. Every (model, prompt) cell of the prompt-selection
# panel is run in both languages, so the pairing is exact.
def paired_scores(frame: pd.DataFrame, index) -> pd.DataFrame:
    paired = frame.pivot_table(index=index, columns="lang", values="f1_ext").dropna()
    paired = paired.reset_index()
    paired["difference"] = paired["EN"] - paired["PT"]
    return paired


def correlation(paired: pd.DataFrame, label: str) -> None:
    r, p = stats.pearsonr(paired["PT"], paired["EN"])
    # Fisher z interval, the usual normal approximation for a correlation.
    z, se = np.arctanh(r), 1 / np.sqrt(len(paired) - 3)
    rho = stats.spearmanr(paired["PT"], paired["EN"]).statistic
    print(f"{label:<26} n={len(paired):>3}  r={r:.3f} "
          f"[{np.tanh(z - 1.96 * se):.3f}, {np.tanh(z + 1.96 * se):.3f}]  "
          f"rho={rho:.3f}  p={p:.1e}")


selection_pairs = paired_scores(SELECTION, ["model", "prompt"])
test_pairs = paired_scores(TEST, ["model", "prompt", "temperature"])
correlation(selection_pairs, "prompt-selection phase")
correlation(test_pairs, "test phase")

prompt-selection phase     n= 72  r=0.946 [0.915, 0.966]  rho=0.929  p=6.2e-36
test phase                 n= 19  r=0.980 [0.946, 0.992]  rho=0.923  p=2.8e-13


In [11]:
# Is there a systematic language gap on top of that agreement, and does it
# differ by model? `amalia` is refitted separately: it abstains on 44.8-59.9% of
# its English queries, and an abstention books every participant of the document
# as a miss, so its English deficit is a parsing artefact rather than a language
# effect.
print(coefficients(fit_ri("difference ~ 1", selection_pairs),
                   "RQ2a  paired EN-PT difference, all models").to_string())
print(coefficients(fit_ri("difference ~ 1", selection_pairs[selection_pairs["model"] != "amalia"]),
                   "RQ2b  paired EN-PT difference, excluding amalia").to_string())

print("\nmean EN-PT difference per model:")
print(selection_pairs.groupby("model")["difference"]
      .agg(["mean", "std", "count"]).round(4).to_string())

--- RQ2a  paired EN-PT difference, all models  (n=72, models=9)
             coef      se  ci_lo   ci_hi       p
Intercept  0.0264  0.0099  0.007  0.0457  0.0076
--- RQ2b  paired EN-PT difference, excluding amalia  (n=64, models=8)
             coef      se   ci_lo   ci_hi    p
Intercept  0.0339  0.0073  0.0196  0.0482  0.0

mean EN-PT difference per model:


                    mean     std  count
model                                  
amalia           -0.0336  0.0263      8
euro_llm_9b       0.0102  0.0480      8
gemini            0.0213  0.0211      8
gemma3_12b        0.0545  0.0297      8
gemma3_4b         0.0537  0.0417      8
gervasio_8b_ptpt  0.0553  0.0505      8
qwen3_14b         0.0368  0.0386      8
qwen3_4b          0.0027  0.0334      8
qwen3_8b          0.0365  0.0274      8


In [12]:
# Does the language effect vary by model? Compare the random-intercept model
# with one that also lets the language slope vary by model. Both are fitted by
# ML, since the fixed-effect structure differs.
without_slope = fit_ri("f1_ext ~ has_exp + has_def + C(task) + C(lang)",
                       SELECTION, reml=False)
with_slope = smf.mixedlm("f1_ext ~ has_exp + has_def + C(task) + C(lang)",
                         data=SELECTION, groups=SELECTION["model"],
                         re_formula="~C(lang)").fit(reml=False, method="lbfgs")
statistic = 2 * (with_slope.llf - without_slope.llf)
print(f"LRT, model-specific language slope: chi2={statistic:.2f}, df=2, "
      f"p={stats.chi2.sf(statistic, 2):.3f}")

LRT, model-specific language slope: chi2=1.57, df=2, p=0.456


**Strongly supported at the headline, and one sub-claim is not.**

*The two languages agree almost completely.* A configuration's Portuguese score
predicts its English score at $r = 0.946$ [0.915, 0.966] across the 72
(model, prompt) cells of the prompt-selection panel, and at $r = 0.980$
[0.946, 0.992] across the 19 test-phase pairs. A configuration that fails in one
language fails in the other. This is the strongest single piece of evidence in
the notebook for RQ2's conceptual reading, and it is quantitative where the
chapter's argument was a comparison of error profiles.

*There is a small systematic English advantage.* The paired difference is
**+0.026 F1** [0.007, 0.046], $p = 0.008$, rising to +0.034 [0.020, 0.048] once
`amalia` - whose English abstentions are a documented parsing artefact - is set
aside. It is real, and it is about a fifth of the example effect.

*The sub-claim that the exceptions attach to particular models is not
supported.* Letting the language effect vary by model does not improve the fit
($\chi^2 = 1.57$, df = 2, $p = 0.46$). The per-model differences range from
$-0.034$ for `amalia` to $+0.055$ for `gervasio_8b_ptpt`, but that spread is
within what noise produces. The chapter should say that the English advantage is
small and general, rather than that its direction depends on the model.

---

## 6. RQ3 - is the ceiling at span identification or at typing?

> **RQ3.** Can LLMs reliably assign participant attributes as defined by the
> Lusa annotation scheme, or is there a systematic ceiling at the span
> identification level, and does this gap differ systematically between
> Portuguese and English?

**The chapter's descriptive answer.** The ceiling is at span identification and
it is not close: no test-phase run loses more to typing than to locating. The
sentence-level regime confirms it by isolating the stages - detection forwards
0.849-0.935 of the sentences it should and typing keeps 0.862-0.899 of the spans
handed to it, while extraction passes on fewer than half the participants that
enter it. The gap does not differ systematically between the languages.

In [13]:
# The funnel has two conditional stages, both proportions on the same
# participants: `recall_ext` is the share of annotated participants located, and
# `accuracy` the share of the located ones typed correctly. Their difference is
# the quantity RQ3 asks about, so it is modelled directly.
funnel = experiments[experiments["accuracy"].notna()
                     & (experiments["extracted"] >= MIN_EXTRACTED)].copy()
funnel["gap"] = funnel["accuracy"] - funnel["recall_ext"]

print(f"runs: {len(funnel)}   locating (recall) = {funnel['recall_ext'].mean():.3f}   "
      f"typing (accuracy) = {funnel['accuracy'].mean():.3f}")
print(f"runs where typing scores below locating: {(funnel['gap'] < 0).sum()} of {len(funnel)}")

rq3 = fit_ri("gap ~ C(lang, Treatment('PT'))", funnel)
print()
print(coefficients(rq3, "RQ3  typing-minus-locating gap").to_string())

print("\nboth stages, by regime and language:")
print(funnel.groupby(["regime", "phase", "lang"])[["recall_ext", "accuracy", "gap"]]
      .mean().round(3).to_string())

runs: 107   locating (recall) = 0.231   typing (accuracy) = 0.859
runs where typing scores below locating: 0 of 107

--- RQ3  typing-minus-locating gap  (n=107, models=10)
                                  coef      se   ci_lo   ci_hi      p
Intercept                       0.6805  0.0392  0.6036  0.7574  0.000
C(lang, Treatment('PT'))[T.EN] -0.0591  0.0215 -0.1012 -0.0170  0.006

both stages, by regime and language:
                                recall_ext  accuracy    gap
regime   phase            lang                             
document prompt_selection EN         0.205     0.861  0.656
                          PT         0.178     0.899  0.721
         test             EN         0.298     0.799  0.502
                          PT         0.293     0.828  0.534
sentence test             EN         0.362     0.873  0.511
                          PT         0.389     0.884  0.495


**Supported, with the second clause qualified.**

*The ceiling is at span identification, decisively.* Typing accuracy exceeds
locating recall by **+0.681** [0.604, 0.757] - the models type 0.86 of what they
find and find 0.23 of what is there. Not one of the 107 runs reverses the
ordering. Across regimes and phases the pattern is identical: locating never
rises above 0.39, typing never falls below 0.79.

*The gap is slightly, but reliably, smaller in English.* The language
coefficient is $-0.059$ [-0.101, -0.017], $p = 0.006$ - about 9% of a gap of
0.68. The chapter's "does not differ systematically" is right in direction, since
the ceiling sits at extraction in both languages, but the strict reading is not
quite accurate: the difference is detectable. It is worth writing as *the gap is
of the same order in both languages, marginally narrower in English*, rather than
as no difference at all.

---

## 7. RQ4 - does cross-lingual assistance help?

> **RQ4.** Can cross-lingual assistance, such as translation-based extraction,
> mitigate these limitations? Does extracting participants from the English
> translation of a Portuguese article reduce prompt sensitivity and improve
> attribute-level reliability, compared with extracting them directly from the
> Portuguese article?

**The chapter's descriptive answer.** It does not. Neither language version is
systematically stronger and the direction depends on the model: the multilingual
models lean slightly to English, the two built with European Portuguese in view
lean the other way, and every margin is smaller than the spread between two
temperatures of one model. What moves the score is the prompting strategy - the
one-shot example, and the move from whole articles to single sentences - not the
language.

In [14]:
# The test phase is the held-out estimate, but the prompt variant is selected per
# (model, language) and two models were selected differently in the two
# languages, which would confound language with prompt. Those two are excluded.
selected_prompt = TEST.groupby(["model", "lang"])["prompt"].first().unstack("lang")
matched_models = selected_prompt.index[selected_prompt["PT"] == selected_prompt["EN"]]
print(f"{len(matched_models)} of {len(selected_prompt)} models carry the same prompt "
      f"in both languages; excluded: {sorted(set(selected_prompt.index) - set(matched_models))}")

matched_test = TEST[TEST["model"].isin(matched_models)]
rq4_language = fit_ri("f1_ext ~ C(lang, Treatment('PT')) + C(temperature)", matched_test)
print()
print(coefficients(rq4_language, "RQ4a  test phase: language and temperature").to_string())

7 of 9 models carry the same prompt in both languages; excluded: ['gemma3_12b', 'qwen3_4b']

--- RQ4a  test phase: language and temperature  (n=39, models=7)
                                  coef      se   ci_lo   ci_hi       p
Intercept                       0.3015  0.0475  0.2083  0.3947  0.0000
C(lang, Treatment('PT'))[T.EN] -0.0001  0.0084 -0.0166  0.0164  0.9909
C(temperature)[T.0.6]          -0.0008  0.0105 -0.0215  0.0199  0.9385
C(temperature)[T.1.0]          -0.0176  0.0105 -0.0383  0.0031  0.0949


In [15]:
# Does the direction depend on how the model was built, as the chapter argues?
PORTUGUESE_ORIENTED = {"euro_llm_9b", "gervasio_8b_ptpt", "amalia"}
by_family = SELECTION.copy()
by_family["family"] = np.where(by_family["model"].isin(PORTUGUESE_ORIENTED),
                               "pt_oriented", "multilingual")

rq4_family = fit_ri(
    "f1_ext ~ has_exp + has_def"
    " + C(lang, Treatment('PT')) * C(family, Treatment('multilingual'))",
    by_family,
)
print(coefficients(rq4_family, "RQ4b  language x model family").to_string())

--- RQ4b  language x model family  (n=152, models=10)
                                                                                      coef      se   ci_lo   ci_hi       p
Intercept                                                                           0.1104  0.0304  0.0508  0.1701  0.0003
C(lang, Treatment('PT'))[T.EN]                                                      0.0334  0.0128  0.0082  0.0586  0.0094
C(family, Treatment('multilingual'))[T.pt_oriented]                                -0.0616  0.0534 -0.1663  0.0432  0.2494
C(lang, Treatment('PT'))[T.EN]:C(family, Treatment('multilingual'))[T.pt_oriented] -0.0228  0.0223 -0.0664  0.0209  0.3074
has_exp                                                                             0.1441  0.0102  0.1241  0.1642  0.0000
has_def                                                                             0.0120  0.0102 -0.0080  0.0321  0.2400


In [16]:
# The competing lever: the regime. Only qwen3_14b was run at sentence level, so
# there is a single group and no random intercept to fit -- this is an ordinary
# fixed-effects contrast over that model's ten test-phase runs.
qwen = experiments[(experiments["model"] == "qwen3_14b") & (experiments["phase"] == "test")]
regime_contrast = smf.ols(
    "f1_ext ~ C(regime, Treatment('document')) + C(lang, Treatment('PT'))", data=qwen
).fit()
interval = regime_contrast.conf_int()
print("RQ4c  regime contrast for qwen3_14b (fixed effects; one model, no grouping factor)")
print(pd.DataFrame({"coef": regime_contrast.params, "se": regime_contrast.bse,
                    "ci_lo": interval[0], "ci_hi": interval[1],
                    "p": regime_contrast.pvalues}).round(4).to_string())
print(f"\ndocument-level F1 spans {qwen[qwen.regime == 'document'].f1_ext.min():.3f}-"
      f"{qwen[qwen.regime == 'document'].f1_ext.max():.3f} over the whole temperature sweep; "
      f"sentence-level {qwen[qwen.regime == 'sentence'].f1_ext.min():.3f}-"
      f"{qwen[qwen.regime == 'sentence'].f1_ext.max():.3f}")

RQ4c  regime contrast for qwen3_14b (fixed effects; one model, no grouping factor)
                                                coef      se   ci_lo   ci_hi       p
Intercept                                     0.3251  0.0076  0.3070  0.3432  0.0000
C(regime, Treatment('document'))[T.sentence]  0.1113  0.0096  0.0887  0.1339  0.0000
C(lang, Treatment('PT'))[T.EN]               -0.0155  0.0094 -0.0376  0.0067  0.1428

document-level F1 spans 0.309-0.327 over the whole temperature sweep; sentence-level 0.394-0.451


**Supported, and the mechanism the chapter proposes is not.**

*Working in English buys nothing on held-out data.* In the test phase the
language coefficient is **-0.0001** [-0.017, 0.016], $p = 0.99$ - as close to
exactly zero as this data can produce. Temperature is flat too: $T = 0.6$ is
$-0.001$ ($p = 0.94$) and $T = 1.0$ is $-0.018$ [-0.038, 0.003], $p = 0.095$.
The +0.026 English advantage of Section 5 is a prompt-selection-phase effect
pooled over all eight variants, including degenerate ones; where each model runs
the prompt chosen for it, it disappears. RQ4's negative answer holds.

*But the split by model family is not statistically supported.* The interaction
between language and model family is $-0.023$ [-0.066, 0.021], $p = 0.31$. The
Portuguese-oriented models do lean the way the chapter describes, but not by
enough to distinguish from noise on nine models. That sentence in the chapter
should be softened to an observation about individual models rather than a
property of how they were built.

*The regime is the lever that does move the score.* For `qwen3_14b`, going from
whole articles to single sentences is worth **+0.111 F1** [0.089, 0.134],
$p < 10^{-6}$, while its language coefficient over the same runs is $-0.016$
($p = 0.14$). The document-level runs span 0.309-0.327 across the entire
temperature sweep; the sentence-level runs sit at 0.394-0.451. The chapter's
closing claim - that how the task is put to the model decides the outcome and
which language it is put in does not - is the one the numbers support most
clearly.

---

## 8. Verdict

Every lever on one scale, as an effect on strict span-extraction F1.

In [17]:
def effect(result, term: str, lever: str, panel: str) -> dict:
    """One row of the summary table, pulled from a fitted result by term name.

    Accepts both the mixed-effects results and the ordinary least-squares
    contrast of Section 7, which expose the same params/bse/conf_int/pvalues
    interface.
    """
    interval = result.conf_int()
    return {
        "lever": lever, "panel": panel,
        "effect": result.params[term],
        "ci_lo": interval[0][term], "ci_hi": interval[1][term],
        "p": result.pvalues[term],
    }


CLS = "C(task, Treatment('ext'))[T.cls]"
levers = pd.DataFrame([
    effect(rq1_extraction, "has_exp", "one-shot example", "prompt selection"),
    effect(regime_contrast, "C(regime, Treatment('document'))[T.sentence]",
           "document -> sentence regime", "test (qwen3_14b)"),
    effect(task_simple["without the example"], CLS,
           "cls task, without example", "prompt selection"),
    effect(fit_ri("difference ~ 1", selection_pairs), "Intercept",
           "language PT -> EN", "prompt selection"),
    effect(task_simple["with the example"], CLS,
           "cls task, with example", "prompt selection"),
    effect(rq4_language, "C(temperature)[T.1.0]", "temperature 0.3 -> 1.0", "test"),
    effect(rq1_extraction, "has_def", "definition", "prompt selection"),
    effect(rq4_language, "C(lang, Treatment('PT'))[T.EN]", "language PT -> EN", "test"),
])
levers["significant"] = np.where(levers["p"] < 0.05, "yes", "no")
print(levers.round(4).to_string(index=False))

                      lever            panel  effect   ci_lo  ci_hi      p significant
           one-shot example prompt selection  0.1333  0.1062 0.1604 0.0000         yes
document -> sentence regime test (qwen3_14b)  0.1113  0.0887 0.1339 0.0000         yes
  cls task, without example prompt selection  0.0731  0.0554 0.0908 0.0000         yes
          language PT -> EN prompt selection  0.0264  0.0070 0.0457 0.0076         yes
     cls task, with example prompt selection  0.0203  0.0026 0.0380 0.0248         yes
     temperature 0.3 -> 1.0             test -0.0176 -0.0383 0.0031 0.0949          no
                 definition prompt selection  0.0077 -0.0194 0.0347 0.5782          no
          language PT -> EN             test -0.0001 -0.0166 0.0164 0.9909          no


### What the random intercept does to the chapter

| RQ | Chapter's claim | Verdict |
| --- | --- | --- |
| RQ1 | The example is the dominant prompt lever | **Confirmed** - +0.133 [0.106, 0.160] |
| RQ1 | Prompt sensitivity does not differ between the languages | **Confirmed** - interaction $p = 0.28$ |
| RQ1 | The definition is a weak lever | **Confirmed, and stronger** - indistinguishable from zero ($p = 0.58$) |
| RQ1 | Task framing is a wash | **Refined** - +0.073 without the example, +0.020 with it; both significant |
| RQ1 | The attribute level is the more stable | **Confirmed** - typing moves by -0.048 against extraction's +0.133 |
| RQ2 | Errors are consistent across the languages | **Confirmed** - $r = 0.95$ (selection), $r = 0.98$ (test) |
| RQ2 | The exceptions attach to particular models | **Not supported** - model-specific language slope $p = 0.46$ |
| RQ3 | The ceiling is at span identification | **Confirmed** - gap +0.681 [0.604, 0.757], 0 of 107 runs reverse it |
| RQ3 | The gap does not differ between the languages | **Qualified** - same order, but reliably narrower in English ($p = 0.006$) |
| RQ4 | Cross-lingual assistance does not help | **Confirmed** - test-phase language effect -0.0001 [-0.017, 0.016] |
| RQ4 | The direction depends on how the model was built | **Not supported** - family interaction $p = 0.31$ |
| RQ4 | Strategy moves the score, language does not | **Confirmed** - regime +0.111 against language -0.000 |

Eight of the twelve claims survive unchanged. Two are not supported and should
be dropped - the model-specific exceptions of RQ2 and the model-family split of
RQ4. One needs qualifying, the RQ3 language clause, and one - task framing -
needs restating as a conditional effect rather than a null one. **None of the
four research-question answers is overturned.**

## 9. All findings on one scale

The levers table above ranks the effects that share a response. This one is the
same idea carried across the four research questions: one row per finding, the
scenario that produces it, the size of that scenario's effect and the p-value
attached to it. It is the draft of the summary table for the chapter.


In [18]:
# Scratch for the chapter's summary table: every finding, the scenario behind it,
# the size of the effect and its p-value. The panel is dropped -- this is the
# general view, and the research question locates each row well enough.
def finding(rq: str, scenario: str, result=None, term: str = "",
            coef: float = np.nan, p: float = np.nan) -> dict:
    """One row of the summary, read off a fitted result or supplied directly.

    `result` and `term` cover the findings that are model coefficients; the two
    keyword arguments cover the likelihood-ratio test, which is not one.
    """
    if result is not None:
        coef, p = result.params[term], result.pvalues[term]
    return {"rq": rq, "scenario": scenario, "coef": coef, "p": p}


# Read the cls term off the fitted model rather than spelling it out, so that
# renaming the column cannot silently break this cell.
CLS_TERM = next(t for t in rq1_task.params.index if t.endswith("[T.cls]"))
LANG_EN = "C(lang, Treatment('PT'))[T.EN]"
FAMILY = "C(family, Treatment('multilingual'))[T.pt_oriented]"
slope_lrt = 2 * (with_slope.llf - without_slope.llf)

findings = pd.DataFrame([
    finding("RQ1", "one-shot example", rq1_extraction, "has_exp"),
    finding("RQ1", "cls task, without the example",
            task_simple["without the example"], CLS_TERM),
    finding("RQ1", "cls task, with the example",
            task_simple["with the example"], CLS_TERM),
    finding("RQ1", "one-shot example x language", rq1_extraction, f"has_exp:{LANG_EN}"),
    finding("RQ1", "definition", rq1_extraction, "has_def"),
    finding("RQ1", "one-shot example, on typing (accuracy points)",
            rq1_typing, "has_exp"),
    finding("RQ2", "language PT -> EN, prompt selection",
            fit_ri("difference ~ 1", selection_pairs), "Intercept"),
    finding("RQ2", "language margin varying by model",
            p=stats.chi2.sf(slope_lrt, 2)),
    finding("RQ3", "typing stands above locating (gap points)", rq3, "Intercept"),
    finding("RQ3", "that gap, English against Portuguese (gap points)", rq3, LANG_EN),
    finding("RQ4", "document -> sentence regime", regime_contrast,
            "C(regime, Treatment('document'))[T.sentence]"),
    finding("RQ4", "temperature 0.3 -> 1.0, test", rq4_language, "C(temperature)[T.1.0]"),
    finding("RQ4", "language PT -> EN, test", rq4_language, LANG_EN),
    finding("RQ4", "language x model family", rq4_family, f"{LANG_EN}:{FAMILY}"),
]).sort_values(["rq", "coef"], ascending=[True, False], na_position="last")

print(findings.round(4).to_string(index=False))
print("\ncoef is in points of F1-ext unless the scenario names another response;")
print("the RQ2 heterogeneity row is a likelihood-ratio test and carries no coefficient.")


 rq                                          scenario    coef      p
RQ1                                  one-shot example  0.1333 0.0000
RQ1                     cls task, without the example  0.0731 0.0000
RQ1                       one-shot example x language  0.0206 0.2795
RQ1                        cls task, with the example  0.0203 0.0248
RQ1                                        definition  0.0077 0.5782
RQ1     one-shot example, on typing (accuracy points) -0.0477 0.0000
RQ2               language PT -> EN, prompt selection  0.0264 0.0076
RQ2                  language margin varying by model     NaN 0.4561
RQ3         typing stands above locating (gap points)  0.6805 0.0000
RQ3 that gap, English against Portuguese (gap points) -0.0591 0.0060
RQ4                       document -> sentence regime  0.1113 0.0000
RQ4                           language PT -> EN, test -0.0001 0.9909
RQ4                      temperature 0.3 -> 1.0, test -0.0176 0.0949
RQ4                           lang

### The four answers in words

Read off the table above, one research question at a time. Every figure is in
points of F1-ext unless another response is named.

**RQ1 — prompt sensitivity across the languages, and span detection against
attribute typing.** The method is strongly sensitive to the prompt, and equally
so in the two languages. The one-shot example is the dominant component, worth
$+0.133$ ($p < 0.001$), and it is worth the same on either side of the language:
the interaction between the example and the language is $+0.021$ ($p = 0.28$), so
nothing here separates Portuguese from English. The definition does nothing that
can be measured, $+0.008$ ($p = 0.58$). Asking for the type alongside the span
helps a bare prompt, $+0.073$ ($p < 0.001$), but little once the example is
present, $+0.020$ ($p = 0.025$): the two components are partly redundant, since
both show the model what an answer should look like. Moving to the attribute
level the picture reverses in scale — the same example costs $-0.048$ of typing
accuracy ($p < 0.001$), reliable but small against a score that sits near 0.88.
**The prompt decides which participants are found, not how the found ones are
labelled, and it does so identically in both languages.**

**RQ2 — are the errors conceptual or language-specific?** Conceptual. A
configuration's Portuguese score predicts its English score almost exactly
(correlations of 0.95 in the prompt-selection phase and 0.98 in the test phase),
so a configuration that fails in one language fails in the other. What separates
the languages on average is small: English stands $+0.026$ above Portuguese
($p = 0.008$). Letting that margin take its own value for each model describes
the runs no better than one value for all of them ($p = 0.46$), so the
model-by-model exceptions visible in the descriptive analysis are not told apart
from noise. **The errors belong to the method, not to the language it is applied
to.**

**RQ3 — can the models assign attributes, or is the ceiling at span
identification?** The ceiling is at span identification, and it is not close.
Across the 107 classifying runs typing stands $0.681$ above locating
($p < 0.001$), and not one run reverses the order of the two stages. The distance
is $0.059$ narrower in English than in Portuguese ($p = 0.006$) — reproducible,
but small beside the distance itself. **Typing is the stage that works; locating
is the stage that limits the method, in both languages.**

**RQ4 — does cross-lingual assistance mitigate the limitation?** It does not.
Where each model runs the variant selected for it, the language is worth
$-0.0001$ ($p = 0.99$), as close to exactly nothing as this data can produce, and
the lean towards one language or the other does not follow how the models were
built, the interaction between language and model family standing at $-0.023$
($p = 0.31$). Decoding temperature is flat as well, $-0.018$ from 0.3 to 1.0
($p = 0.095$). What does move the score is the granularity of the query: putting
the same articles to the same model one sentence at a time is worth $+0.111$
($p < 0.001$), second only to the example. **How the task is put to the model
decides the outcome; which of the two languages it is put in does not.**
